In [1]:
import logging
import os
from typing import TYPE_CHECKING

import numpy as np
from scipy.spatial.transform import Rotation

from mascaf import (
    BasisOptimizerOptions,
    CableFitter,
    FitOptions,
    MeshManager,
    SkeletonGraph,
    Validation,
)
from swctools import SWCModel, plot_model

if TYPE_CHECKING:
    import plotly.graph_objects as go  # type: ignore

logging.basicConfig(level=logging.INFO)

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [2]:
# per-spine parameters

def get_ts_pipeline_params(idx: int) -> dict:
    qst = 0.5  # for all
    mcst = 5  # for all

    fig_width = 800
    fig_height = 600

    rotations = {
        1: [0, -30, 20], 
        2: [-40, 0, 0], 
        3: [60, 0, 20],
        4: [-30, 40, 50],
        21: [0, 30, 95],
        24: [0, 0, 0],
        48: [0, 0, 0],
        67: [0, 0, 0],
        76: [0, 0, 0],
    }

    zooms = {
        1: 1.0, 
        2: 1.0, 
        3: 1.0, 
        4: 1.0, 
        21: 1.0, 
        24: 1.0, 
        48: 1.0, 
        67: 1.0, 
        76: 1.0,
    }

    pruning_fractions = {
        1: 0.2, 
        2: 0.2, 
        3: 0.2, 
        4: 0.2, 
        21: 0.2, 
        24: 0.2, 
        48: 0.2, 
        67: 0.2, 
        76: 0.2,
    }

    # max_edge_lengths = {
    #     1: 200, 
    #     2: 50, 
    #     3: 150, 
    #     4: 200, 
    #     21: 200, 
    #     24: 200, 
    #     48: 200, 
    #     67: 200, 
    #     76: 200,
    # }

    optimizer_options = BasisOptimizerOptions(
        do_pruning=True,
        pruning_min_length_fraction=pruning_fractions[idx],
        do_snapping=True,
        do_forcing=True,
        n_rays=6,
        max_iterations=20,
        step_size=1.0,
        smoothing_weight=0.1,
        preserve_terminal_nodes=True,
        preserve_branch_nodes=False,
    )

    rot = Rotation.from_euler("xyz", rotations[idx], degrees=True)
    zoom = zooms[idx]
    eye_coord = rot.apply(np.array([1.0, 1.0, 1.0])) * zoom

    return {
        "object_name": f"TS{idx}",
        "qst": qst,
        "mcst": mcst,
        "eye_coord": eye_coord,
        # "fit_options": fit_options,
        "fig_width": fig_width,
        "fig_height": fig_height,
        # "max_edge_length": max_edge_lengths[idx],
        "max_edge_length": None,
        "optimizer_options": optimizer_options,
    }

pdf_scale = 2

In [3]:
spine_idx = 21
params = get_ts_pipeline_params(spine_idx)

mesh_path = f"../data/mesh/processed/{params['object_name']}.obj"
mm = MeshManager(mesh_path=mesh_path)
model_length = mm.bounding_box_diagonal()
params["max_edge_length"] = int(model_length * 0.09)

polylines_name = f"TS{spine_idx}_qst{params['qst']}_mcst{params['mcst']}"
skeleton = SkeletonGraph.from_txt(
    f"../data/mcf_skeletons/{polylines_name}.polylines.txt"
)

fig_out_dir = f"../viz/ts{spine_idx}"
os.makedirs(fig_out_dir, exist_ok=True)

# FIGURE: mesh
mesh_fig: "go.Figure" = mm.visualize_mesh_3d(skel=None, show_axes=False, title="")

eye_coord = params["eye_coord"]

mesh_fig.update_layout(
    scene=dict(
        camera=dict(
            eye={"x": eye_coord[0], "y": eye_coord[1], "z": eye_coord[2]},
            projection=dict(type="perspective"),
        ),
        aspectmode="data",  # 'cube', 'auto', 'manual'
    )
)
mesh_fig.write_image(
    f"{fig_out_dir}/TS{spine_idx}_mesh.pdf",
    format="pdf",
    engine="kaleido",
    width=600,
    height=450,
    scale=pdf_scale,
)
mesh_fig.show()

# FIGURE: mesh with skeleton
mesh_skel_fig: "go.Figure" = mm.visualize_mesh_3d(
    skel=skeleton, show_axes=False, title=""
)
mesh_skel_fig.update_layout(
    scene=dict(
        camera=dict(
            eye={"x": eye_coord[0], "y": eye_coord[1], "z": eye_coord[2]},
            projection=dict(type="perspective"),
        ),
        aspectmode="data",  # 'cube', 'auto', 'manual'
    )
)
mesh_skel_fig.write_image(
    f"{fig_out_dir}/TS{spine_idx}_mesh_skel.pdf",
    format="pdf",
    engine="kaleido",
    width=600,
    height=450,
    scale=pdf_scale,
)
mesh_skel_fig.show()

INFO:mascaf.mesh:Loaded mesh: 2017 vertices, 4040 faces
C:\Users\MainUser\AppData\Local\Temp\ipykernel_35272\1994545073.py:31: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  mesh_fig.write_image(
INFO:choreographer.browsers.chromium:Chromium init'ed with kwargs {}
INFO:choreographer.browsers.chromium:Found chromium path: C:\Program Files (x86)\Google\Chrome\Application\chrome.exe
INFO:choreographer.utils._tmpfile:Temp directory created: C:\Users\MainUser\AppData\Local\Temp\tmp89nqk72d.
INFO:choreographer.browser_async:Opening browser.
INFO:choreographer.utils._tmpfile:Temp directory created: C:\Users\MainUser\AppData\Local\Temp\tmpsoy02ota.
INFO:choreographer.browsers.chromium:Temporary directory at: C:\Users\MainUser\AppData\Local\Temp\tmpsoy02ota
INFO:kaleido.kaleido:Conforming 1 to file:///C:/Users/MainUser/AppData/Local/Temp/tmp89nqk72d/index.html
INFO:kaleido.

C:\Users\MainUser\AppData\Local\Temp\ipykernel_35272\1994545073.py:54: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  mesh_skel_fig.write_image(
INFO:choreographer.browsers.chromium:Chromium init'ed with kwargs {}
INFO:choreographer.browsers.chromium:Found chromium path: C:\Program Files (x86)\Google\Chrome\Application\chrome.exe
INFO:choreographer.utils._tmpfile:Temp directory created: C:\Users\MainUser\AppData\Local\Temp\tmp0qmvvjtr.
INFO:choreographer.browser_async:Opening browser.
INFO:choreographer.utils._tmpfile:Temp directory created: C:\Users\MainUser\AppData\Local\Temp\tmpjlkj1sgx.
INFO:choreographer.browsers.chromium:Temporary directory at: C:\Users\MainUser\AppData\Local\Temp\tmpjlkj1sgx
INFO:kaleido.kaleido:Conforming 1 to file:///C:/Users/MainUser/AppData/Local/Temp/tmp0qmvvjtr/index.html
INFO:kaleido.kaleido:Getting tab from queue (has 1)
INFO:kaleido

In [4]:
swc_out_dir = f"../data/swc/current/{polylines_name}"
swc_filepath = f"{swc_out_dir}/TS{spine_idx}_mel{params["max_edge_length"]}.swc"

# check if directory exists, if not create it
if not os.path.exists(swc_out_dir):
    os.makedirs(swc_out_dir)

fitter = CableFitter(options=FitOptions(
        max_edge_length=params['max_edge_length'],
        radius_strategy="equivalent_area",
        section_probe_eps=1e-4,
        section_probe_tries=3,
        multi_tangent_reduction="mean",
        basis_optimizer_options=params['optimizer_options'],
    ))

morph = fitter.fit(
    mm.mesh,
    skeleton,
)
# write swc to file
morph.to_swc_file(swc_filepath)
# validation
validator = Validation(mm, skeleton, morph)
validator.full_validation()

model = SWCModel.from_swc_file(swc_filepath)
model.print_attributes(node_info=False, edge_info=False)
title = f"TS{spine_idx}_mel{params['max_edge_length']}"
morph_fig = plot_model(
    swc_model=model,
    slider=False,
    title="",
    width=800,
    height=600,
    show_axes=False,
)
morph_fig.update_layout(
    scene=dict(
        camera=dict(
            eye={"x": eye_coord[0], "y": eye_coord[1], "z": eye_coord[2]},
            projection=dict(type="perspective"),
        ),
        aspectmode="data",  # 'cube', 'auto', 'manual'
    )
)
morph_filename = (
    f"{fig_out_dir}/TS{spine_idx}_mel{params['max_edge_length']}_" f"morph.pdf"
)
morph_fig.write_image(
    morph_filename,
    format="pdf",
    engine="kaleido",
    width=600,
    height=450,
    scale=pdf_scale,
)
morph_fig.show()

INFO:mascaf.cable_fitting:Starting cable fit with 251 skeleton nodes, 251 skeleton edges, 2017 mesh vertices, and max_edge_length=78
INFO:mascaf.cable_fitting:Optimizing morphology basis before radius fitting
INFO:mascaf.basis_optimizer:Starting basis optimization...
INFO:mascaf.basis_optimizer:  Nodes: 34
INFO:mascaf.basis_optimizer:Phase 0 - Pruning
INFO:mascaf.basis_optimizer:  Removing branches with length < 21.2637
INFO:mascaf.basis_optimizer:Phase 1 - Snapping: 0 nodes outside mesh
INFO:mascaf.basis_optimizer:Phase 2 - Forcing: max 20 iterations
INFO:mascaf.basis_optimizer:  Iteration 0: avg movement = 0.623267
INFO:mascaf.basis_optimizer:  Iteration 1: avg movement = 0.615867
INFO:mascaf.basis_optimizer:  Iteration 2: avg movement = 0.611134
INFO:mascaf.basis_optimizer:  Iteration 3: avg movement = 0.611229
INFO:mascaf.basis_optimizer:  Iteration 4: avg movement = 0.609861
INFO:mascaf.basis_optimizer:  Iteration 5: avg movement = 0.612980
INFO:mascaf.basis_optimizer:  Iteration 

SWCModel: nodes=33, edges=32, components=1, cycles=0, branch_points=8, roots=1, leaves=9, self_loops=0, density=0.0606


INFO:kaleido.kaleido:Conforming 1 to file:///C:/Users/MainUser/AppData/Local/Temp/tmpvd4pb100/index.html
INFO:kaleido.kaleido:Getting tab from queue (has 1)
INFO:kaleido.kaleido:Got 95C7
INFO:kaleido.kaleido:Reloading tab 95C7 before return.
INFO:kaleido.kaleido:Putting tab 95C7 back (queue size: 0).
INFO:kaleido.kaleido:Waiting for all cleanups to finish.
INFO:kaleido.kaleido:Exiting Kaleido.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:choreographer.browser_async:Closing browser.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:choreographer.browser_async:Closing browser.
INFO:kaleido.kaleido:Cancelling tasks.
INFO:kaleido.kaleido:Exiting Kaleido/Choreo.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:kaleido.kaleido:Cancelling tasks.

In [5]:
morph.scale_radii_to_match_mesh(
    mm.mesh, metric="surface_area", account_for_overlaps=False
)

# save normalized to file
swc_filepath = f"{swc_out_dir}/TS{spine_idx}_mel{params["max_edge_length"]}_norm.swc"
morph.to_swc_file(swc_filepath)

# load and plot
swc_model = SWCModel.from_swc_file(swc_filepath)
swc_model.print_attributes(node_info=False, edge_info=False)
title = f"TS{spine_idx}_s{params["max_edge_length"]}"
norm_fig: "go.Figure" = plot_model(
    swc_model=swc_model,
    slider=False,
    title="",
    width=800,
    height=600,
    show_axes=False,
)
norm_fig.update_layout(
    scene=dict(
        camera=dict(
            eye={"x": eye_coord[0], "y": eye_coord[1], "z": eye_coord[2]},
            projection=dict(type="perspective"),
        ),
        aspectmode="data",  # 'cube', 'auto', 'manual'
    )
)
norm_filename = (
    f"{fig_out_dir}/TS{spine_idx}_mel{params['max_edge_length']}_" f"morph_norm.pdf"
)
norm_fig.write_image(
    norm_filename,
    format="pdf",
    engine="kaleido",
    width=600,
    height=450,
    scale=pdf_scale,
)
norm_fig.show()

validator = Validation(mm, skeleton, morph)
validator.full_validation()

INFO:swctools.io:parse_swc start strict=True validate_reconnections=True float_tol=1e-09
INFO:swctools.io:parse_swc done records=33 reconnections=1 header=5
INFO:swctools.model:SWCModel.from_parse_result records=33 reconnections=1 header=5
INFO:swctools.model:SWCModel.from_swc_file built nodes=33 edges=32 strict=True validate_reconnections=True
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:swctools.geometry:batch_frusta count=32 sides=16 end_caps=False verts=1024 faces=1024
INFO:swctools.geometry:FrustaSet.from_swc_model edges=32 sides=16 end_caps=False
INFO:swctools.viz:plot_model slider=False frusta=32 show_frusta=True show_centroid=True
C:\Users\MainUser\AppData\Local\Temp\ipykernel_35272\2178072094.py:33: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  norm_fig.write_image(
INF

SWCModel: nodes=33, edges=32, components=1, cycles=0, branch_points=8, roots=1, leaves=9, self_loops=0, density=0.0606


INFO:kaleido.kaleido:Conforming 1 to file:///C:/Users/MainUser/AppData/Local/Temp/tmp02iu095r/index.html
INFO:kaleido.kaleido:Getting tab from queue (has 1)
INFO:kaleido.kaleido:Got 2E6F
INFO:kaleido.kaleido:Reloading tab 2E6F before return.
INFO:kaleido.kaleido:Putting tab 2E6F back (queue size: 0).
INFO:kaleido.kaleido:Waiting for all cleanups to finish.
INFO:kaleido.kaleido:Exiting Kaleido.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:choreographer.browser_async:Closing browser.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:choreographer.browser_async:Closing browser.
INFO:kaleido.kaleido:Cancelling tasks.
INFO:kaleido.kaleido:Exiting Kaleido/Choreo.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:kaleido.kaleido:Cancelling tasks.

INFO:mascaf.validation:Initialized Validation from MorphologyGraph
INFO:mascaf.validation:  Mesh: 2017 vertices, 4040 faces
INFO:mascaf.validation:  Skeleton: 251 nodes, 251 edges
INFO:mascaf.validation:  MorphologyGraph: 32 nodes, 32 edges
INFO:mascaf.validation:Validation Results, account_for_overlaps=False:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       7358716.0420
INFO:mascaf.validation:---- Morphology volume: 8539645.2802
INFO:mascaf.validation:---- Ratio:             1.1605
INFO:mascaf.validation:---- Error:             1180929.2382
INFO:mascaf.validation:---- Relative error:    16.05%
INFO:mascaf.validation:-- Surface Area Comparison:
INFO:mascaf.validation:---- Mesh area:         477408.7914
INFO:mascaf.validation:---- Morphology area:   477408.7914
INFO:mascaf.validation:---- Ratio:             1.0000
INFO:mascaf.validation:---- Error:             -0.0000
INFO:mascaf.validation:---- Relative error:    -0.00%
INFO:mascaf.validation:

In [6]:
# compare morphology basis to original skeleton

skel_pointset = skeleton.to_point_set()

vs_fig: "go.Figure" = plot_model(
    swc_model=swc_model,
    opacity=0.2,
    title="",
    width=800,
    height=600,
    show_axes=False,
    point_set=skel_pointset,
    point_color="red",
    point_size=model_length * 0.002,
)
vs_fig.update_layout(
    scene=dict(
        camera=dict(
            eye={"x": eye_coord[0], "y": eye_coord[1], "z": eye_coord[2]},
            projection=dict(type="perspective"),
        ),
        aspectmode="data",  # 'cube', 'auto', 'manual'
    )
)
vs_filename = (
    f"{fig_out_dir}/TS{spine_idx}_mel{params['max_edge_length']}_" f"morph_vs_skel.pdf"
)
vs_fig.write_image(
    vs_filename,
    format="pdf",
    engine="kaleido",
    width=600,
    height=450,
    scale=pdf_scale,
)
vs_fig.show()

INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:swctools.geometry:batch_spheres count=-1 stacks=6 slices=12 verts=15562 faces=30120
INFO:swctools.geometry:PointSet.from_points n=251 base_radius=1.0 stacks=6 slices=12
INFO:swctools.geometry:batch_frusta count=32 sides=16 end_caps=False verts=1024 faces=1024
INFO:swctools.geometry:FrustaSet.from_swc_model edges=32 sides=16 end_caps=False
INFO:swctools.geometry:batch_spheres count=251 stacks=6 slices=12 verts=15562 faces=30120
INFO:swctools.geometry:PointSet.scaled radius_scale=1.7414973394725963
INFO:swctools.viz:plot_model slider=False frusta=32 show_frusta=True show_centroid=True
C:\Users\MainUser\AppData\Local\Temp\ipykernel_35272\932110681.py:28: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  vs_fig.write_image(
INFO:choreographer